# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field @id's
print("Record sets and their fields (by @id):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we'll use the first record set (adjust as needed)
all_record_sets = list(dataset.record_sets)
record_sets_ids = [rs.id for rs in all_record_sets]
dataframes = {}

# Load all record sets into dataframes
for rs in all_record_sets:
    print(f"Loading records for record set: {rs.name} (@id: {rs.id})")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Rows: {df.shape[0]}")
    except Exception as e:
        print(f"  Failed to load data for record set {rs.id}: {e}")

# Pick one record set for further analysis (first one)
if len(record_sets_ids) > 0:
    main_record_set_id = record_sets_ids[0]
    print(f"\nMain record set for analysis: {main_record_set_id}")

    main_df = dataframes[main_record_set_id]
    print("Columns:", main_df.columns.tolist())
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric and a grouping field based on prior overview
numeric_field = None
group_field = None
possible_numeric_fields = []
df = main_df

# Try to auto-detect a numeric column (fall back to manual)
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col].dropna()):
        possible_numeric_fields.append(col)

if len(possible_numeric_fields) == 0:
    print("No numeric fields (with recognized dtypes) found. Listing all columns for manual inspection.")
    print(df.columns)
else:
    print("Detected numeric fields:")
    print(possible_numeric_fields)
    numeric_field = possible_numeric_fields[0]

# Try to pick a group field (typically string column with few unique values)
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < 16 and col != numeric_field:
        group_field = col
        break
print(f"Selected numeric_field: {numeric_field}")
print(f"Selected group_field: {group_field}")

if numeric_field is None:
    print("No numeric field selected, skipping EDA.")
else:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    normalized_field = f"{numeric_field}_normalized"
    filtered_df[normalized_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_field]].head())

    # Group, if grouping field exists
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (means):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df, showmeans=True)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we've demonstrated how to load, inspect, and analyze the mass spectrometry dataset of extracellular vesicles from stimulated human mast cells using `mlcroissant`. We've loaded the metadata, identified record sets and field `@id`s, extracted tabular data, performed basic EDA (including normalization, filtering, grouping), and visualized distributions and relationships. Use the referenced field and record set `@id`s for consistency in all downstream analysis. Continue with domain-specific statistical and machine learning analysis as relevant to your objectives.*